In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%%RecordEvent
import numpy as np
import pandas as pd
from pathlib import Path
from utils.benchmarks import BENCHMARKS_TO_PATHS


In [ ]:
### cell 0 ###

benchmark_name = "nyc-flight"
factor = 2
flights_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nyc_flights.csv"
)
flights_df = pd.concat([flights_df] * factor, ignore_index=True)

In [ ]:
### cell 1 ###

flights_df.shape, flights_df.columns, flights_df.dtypes

In [ ]:
### cell 2 ###

flights_df.dest.unique()
flights_df.head(10)

In [ ]:
### cell 3 ###
flights_df["dest"].value_counts().loc[["SEA"]]

In [ ]:
### cell 4 ###

flights_df.loc[flights_df.dest.isin(["SEA"]), "carrier"].value_counts()

In [ ]:
### cell 5 ###
# include nulls so it matches len(unique()) behavior
flights_df.loc[flights_df["dest"] == "SEA", "tailnum"].nunique(dropna=False)

In [ ]:
### cell 6 ###

flights_df.loc[flights_df.dest == "SEA", "arr_delay"].mean()

In [ ]:
### cell 7 ###

# cell 7 (optimized for cudf)
f = flights_df.loc[flights_df['dest'] == 'SEA', 'origin']\
               .value_counts(normalize=True)
f.loc['EWR'], f.loc['JFK']

In [ ]:
### cell 8 ###

df = flights_df.groupby(["month", "day"], as_index=False).agg({"dep_delay": np.mean})
df2 = flights_df.groupby(["month", "day"], as_index=False).agg({"arr_delay": np.mean})
df.loc[df["dep_delay"].idxmax()], df2.loc[df2["arr_delay"].idxmax()]

In [ ]:
### cell 9 ###

df

In [ ]:
### cell 10 ###

df = flights_df.groupby(["day", "month"], as_index=False).agg(
    {"arr_delay": np.mean, "dep_delay": np.mean}
)
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
df.sort_values("total_delay", ascending=False).head(1)

In [ ]:
### cell 11 ###

ds = flights_df.groupby("month")["dep_delay"].mean()
ds

In [ ]:
### cell 12 ###

dt = flights_df.groupby("hour")["dep_delay"].mean().dropna()
dt

In [ ]:
### cell 13 ###

df = flights_df
df["speed"] = df["distance"] / df["air_time"]
df[df["speed"] == df.speed.max()]

In [ ]:
### cell 14 ###

# Compute group sizes on GPU
df = flights_df.groupby(["carrier", "flight", "dest"]).size() \
               .reset_index(name="Size")
# Filter for groups with Size == 365 using a single GPU boolean-index operation
df_filtered = df[df["Size"] == 365]
# Loop only over the matching rows (usually far fewer than the full df)
for carrier, flight, dest in zip(
        df_filtered["carrier"],
        df_filtered["flight"],
        df_filtered["dest"]
    ):
    print(f"Carrier: {carrier}, Flight: {flight}, Destination: {dest}")
# Preserve the loop variable i from the original code (last index in df)
i = df.index[-1]

In [ ]:
### cell 15 ###
# Do the entire group‐by and count on the GPU without resetting a MultiIndex
# Use a non‐grouping column (e.g. 'year') so that all three keys stay in the result
counts = (
    flights_df
    .groupby(["carrier", "flight", "dest"], as_index=False)
    .year.count()
    .rename(columns={"year": "size"})
)

# Filter for routes that flew all 365 days, then project the grouping columns
complete_year = counts.loc[counts["size"] == 365, ["carrier", "flight", "dest"]]
complete_year

In [ ]:
### cell 16 ###

# Optimized for cudf

df = flights_df[flights_df["month"] == 6]
# group and aggregate on GPU
df = df.groupby("carrier", as_index=False).agg({
    "arr_delay": "mean",
    "dep_delay": "mean"
})
# vectorized addition on GPU
df["total_delay"] = df["arr_delay"] + df["dep_delay"]
# find index of minimum total_delay on GPU
idx = df["total_delay"].idxmin()
# print the carrier with the smallest total_delay
print(df["carrier"].iloc[idx], df["total_delay"].iloc[idx])

# return the dataframe
df

In [ ]:
### cell 17 ###

weather_df = pd.read_csv(
    Path(BENCHMARKS_TO_PATHS[benchmark_name]).parent / "input" / "nyc_weather.csv"
)
df = flights_df
df_c = pd.merge(df, weather_df, on=["month", "day", "hour", "origin"])
df_c.head(10)